# Klustring av amerikanska aktier

## 1. Problem och syfte

Investerare som vill sprida sin risk väljer ofta aktier från olika sektorer. Men aktier från olika sektorer kan ändå bete sig likt, till exempel röra sig lika mycket när marknaden svänger eller ha liknande lönsamhet och skuldsättning.

**Syfte:** gruppera cirka 2 000 amerikanska aktier efter bolagens räkenskaper och aktiernas kursbeteende, och undersöka om grupperna följer sektorindelningen eller om det finns några andra mönster.

**Varför maskininlärning?** Med tusentals aktier och ett tiotal variabler per aktie går det inte att jämföra alla aktier med varandra för hand. Klustring hittar grupper av liknande aktier automatiskt, utan att vi bestämmer i förväg vilka grupperna ska vara.

**Slutprodukt:** en app där man väljer en aktie och ser vilken grupp den tillhör och vilka aktier som liknar den mest.

---

> ⚠️ **Obs! Inga investeringsråd.**
> Analysen beskriver historiskt beteende. Att aktier har betett sig likt tidigare garanterar inte att de gör det i framtiden.

## 2. Data och källor

Data hämtas från tre öppna API:er med skriptet `src/fetch_data.py` och sparas som CSV i `data/raw/`, så att notebooken kan köras utan nya API-anrop.

| Källa | Innehåll |
|---|---|
| Nasdaq | Alla cirka 7 250 amerikanska aktier med sektor, bransch, land och börsvärde |
| SEC EDGAR | Räkenskaper för 2025: omsättning, resultat, FoU, tillgångar, skulder, eget kapital, kassa |
| Yahoo Finance | Tre års dagliga kurser (cirka 750 handelsdagar) samt S&P 500-indexet (SPY) |

**Avgränsning:** endast aktier med börsvärde över 300 miljoner dollar (gränsen mellan mikrobolag och småbolag), komplett räkenskapsdata och tre års kurshistorik.

In [2]:
from pathlib import Path

# Importera verktyg för numeriska beräkningar, tabeller och diagram
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visa max 50 kolumner
pd.set_option("display.max_columns", 50)
RAW_DIR = Path("..") / "data" / "raw"

# Läs in de tre rådatafilerna
nasdaq = pd.read_csv(RAW_DIR / "nasdaq_aktier.csv")
sec = pd.read_csv(RAW_DIR / "sec_rakenskaper.csv")
prices = pd.read_csv(RAW_DIR / "kurser.csv", index_col=0, parse_dates=True)

# Skriver ut antalet aktier, kolumnantal
print(f"Nasdaq: {nasdaq.shape[0]} aktier, {nasdaq.shape[1]} kolumner")
print(f"SEC:    {sec.shape[0]} tickers, {sec.shape[1]} kolumner")
print(f"Kurser: {prices.shape[0]} dagar x {prices.shape[1]} aktier "
      f"({prices.index.min():%Y-%m-%d} till {prices.index.max():%Y-%m-%d})")

Nasdaq: 7264 aktier, 7 kolumner
SEC:    10404 tickers, 10 kolumner
Kurser: 751 dagar x 3609 aktier (2023-11-17 till 2026-11-16)


In [3]:
display(nasdaq.head())        # de 5 första raderna i aktielistan
display(sec.head())           # de 5 första raderna i räkenskaperna
display(prices.iloc[:5, :5])  # de 5 första dagarna × de 5 första aktierna

,ticker,name,sector,industry,country,ipoyear,marketCap
0,A,Agilent Technologies Inc. Common Stock,Industrials,Biotechnology: Laboratory Analytical Instruments,United States,1999.0,4.908099e+10
1,AA,Alcoa Corporation Common Stock,Industrials,Aluminum,United States,2016.0,1.048488e+10
2,AAC,Ares Acquisition Corporation III Class A Ordin...,Industrials,Metal Fabrications,NaN,2026.0,4.917166e+08
3,AAC-WS,Ares Acquisition Corporation III Redeemable wa...,Industrials,Metal Fabrications,NaN,NaN,0.000000e+00
4,AACG,ATA Creativity Global American Depositary Shares,Real Estate,Other Consumer Services,China,2008.0,5.135486e+07


,ticker,cik,net_income,operating_income,rnd_expense,assets,liabilities,equity,cash,revenue
0,AAPL,320193,1.120100e+11,1.330500e+11,3.455000e+10,3.792970e+11,2.911070e+11,8.819000e+10,4.531700e+10,4.161610e+11
1,NVDA,1045810,1.200670e+11,1.303870e+11,1.849700e+10,2.068030e+11,4.951000e+10,1.572930e+11,1.060500e+10,2.159380e+11
2,MSFT,789019,1.018320e+11,1.285280e+11,3.248800e+10,6.653020e+11,2.744270e+11,3.908750e+11,2.429600e+10,2.817240e+11
3,GOOGL,1652044,1.321700e+11,1.290390e+11,6.108700e+10,5.952810e+11,1.800160e+11,4.152650e+11,3.070800e+10,4.028360e+11
4,AMZN,1018724,7.767000e+10,7.997500e+10,NaN,8.180420e+11,NaN,4.110650e+11,8.681000e+10,7.169240e+11


,A,AA,AAC,AADX,AAL
Date,,,,,
2023-11-17,110.653252,25.388779,NaN,NaN,12.29
2023-11-20,111.464935,25.630856,NaN,NaN,12.40
2023-11-21,121.185600,25.553391,NaN,NaN,12.13
2023-11-22,121.254044,25.543711,NaN,NaN,12.31
2023-11-24,123.826035,25.408146,NaN,NaN,12.31


**Iakttagelser:** En första titt visar att allt i datan inte är vanliga aktier. Här finns till exempel skalbolag utan verksamhet (AAC), warrants (AAC-WS) och nyligen noterade bolag som saknar tre års kurshistorik. Vissa bolag saknar också räkenskapsposter (AMZN). Det rensas bort i nästa steg.

## 3. EDA

### 3.1 Urval och datarensning

Tabellerna kopplas ihop via ticker och filtreras i fyra steg:

1. **Börsvärde över 300 Miljoner USD:** mikrobolag tas bort eftersom de ofta handlas lite och har ryckiga kurser.
2. **Ett bolag per rad:** vissa bolag har flera aktieslag med samma räkenskaper (t.ex. har bolaget Alphabet både GOOG och GOOGL). Vi behåller aktien med högst börsvärde, så att det inte blir dubbletter.
3. **Komplett räkenskapsdata:** omsättning, nettoresultat, tillgångar och skulder behövs för att räkna ut nyckeltalen. Saknas skulder räknar vi fram dem (tillgångar − eget kapital). Bolag som ändå saknar någon uppgift tas bort.
4. **Tre års kurshistorik:** minst 95 % av handelsdagarna måste ha en kurs. Nyare börsnoteringar faller bort.

In [7]:
# 1. Börsvärde över 300 miljoner USD
df = nasdaq[nasdaq["marketCap"] > 300_000_000].copy()
print(f"1. Börsvärde > 300 miljoner USD:     {len(df)} aktier")

# Koppla på räkenskaper från SEC (bara aktier som finns i båda tabellerna)
df = df.merge(sec, on="ticker", how="inner")

# 2. Ett bolag per rad: behåll aktieslaget med högst börsvärde för att rensa dubbletter
df = df.sort_values("marketCap", ascending=False).drop_duplicates("cik")
print(f"2. Finns hos SEC, ett per bolag: {len(df)} aktier")

# Kollar så Skulder saknas ibland men kan räknas fram: skulder = tillgångar - eget kapital
df["liabilities"] = df["liabilities"].fillna(df["assets"] - df["equity"])

# 3. Komplett räkenskapsdata
required = ["revenue", "net_income", "assets", "liabilities"]
df = df.dropna(subset=required)
print(f"3. Komplett räkenskapsdata:  {len(df)} aktier")

# 4. Tre års kurshistorik: kurs minst 95 % av handelsdagarna
coverage = prices.notna().mean()                  # andel dagar med kurs, per aktie
full_history = coverage[coverage >= 0.95].index   # aktier som klarar kravet
df = df[df["ticker"].isin(full_history)].reset_index(drop=True)
print(f"4. Tre års kurshistorik:     {len(df)} aktier")

1. Börsvärde > 300 miljoner USD:     3608 aktier
2. Finns hos SEC, ett per bolag: 3391 aktier
3. Komplett räkenskapsdata:  2173 aktier
4. Tre års kurshistorik:     2034 aktier
